# Module 2 — Return Calculations

Exploratory notebook for `analysis/returns.py`. Given the adjusted-close
prices from Module 1, compute **daily**, **cumulative**, **annualized**, and
**rolling** returns.

**Key concepts:** NumPy vectorized math, `pandas.pct_change()`, log returns
(`np.log` / `np.exp`), and rolling windows.

> Convention (see `SPEC.md`): explore here first, then refactor the working
> logic into `analysis/returns.py`. This notebook is kept as an honest
> artifact of the process.

## Setup

In [ ]:
import sys
from pathlib import Path

# Make the project root importable when running from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from analysis import fetch, returns
from analysis.config import DEFAULT_TICKERS, DEFAULT_START

# Pull from the local cache written by Module 1 (network only on a miss).
prices = fetch.load_or_fetch(DEFAULT_TICKERS, start=DEFAULT_START)
prices.tail()

## 1. Daily returns

Simple percentage returns, `(P_t - P_{t-1}) / P_{t-1}`. `pandas` gives us
`.pct_change()`; `returns.daily_returns` wraps it and drops the leading
all-NaN row.

In [ ]:
daily = returns.daily_returns(prices)
daily.head()

In [ ]:
# The same thing by hand with NumPy — vectorized, no Python loop.
values = prices.to_numpy()
manual = (values[1:] - values[:-1]) / values[:-1]
np.allclose(manual, daily.to_numpy(), equal_nan=True)

## 2. Simple vs log returns — when does it matter?

Log returns are `ln(P_t / P_{t-1})`. They are *additive over time* (daily log
returns sum to the period log return), which makes them convenient for
aggregation and statistics. For small daily moves they're almost identical to
simple returns; the gap grows as moves get bigger.

In [ ]:
logr = returns.log_returns(prices)

compare = pd.DataFrame({
    "simple": daily["VTI"],
    "log": logr["VTI"],
    "diff": daily["VTI"] - logr["VTI"],
})
compare.describe()

In [ ]:
# The difference is tiny day-to-day but scales with the size of the move.
ax = compare.plot.scatter(x="simple", y="diff", s=4, figsize=(8, 4),
                          title="simple - log return vs. size of move (VTI)")
plt.tight_layout()

## 3. Cumulative returns — growth of $1

`(1 + r).cumprod() - 1` chains the daily returns into total return to date.
Plotting all tickers on one axis is the classic comparison chart.

In [ ]:
cum = returns.cumulative_returns(daily)
cum.plot(figsize=(10, 5), title="Cumulative return since 2013")
plt.ylabel("Total return")
plt.axhline(0, color="black", lw=0.5)
plt.tight_layout()

## 4. Annualized return (CAGR)

Geometric annualized return: compound every daily return into one growth
factor, then rescale to a per-year rate using ~252 trading days/year.

In [ ]:
ann = returns.annualized_return(daily)
ann.sort_values(ascending=False).map(lambda x: f"{x:.2%}")

## 5. Rolling 1-year return

A 252-day trailing window shows how performance varied across time instead of
as a single summary number. Watch 2020 and 2022 swing the bond/stock lines.

In [ ]:
roll = returns.rolling_return(daily, window=252)
roll.plot(figsize=(10, 5), title="Rolling 1-year return")
plt.ylabel("Trailing 1y return")
plt.axhline(0, color="black", lw=0.5)
plt.tight_layout()

In [ ]:
# Best and worst rolling 1-year window each ticker ever had (in this range).
pd.DataFrame({
    "best_1y": roll.max(),
    "worst_1y": roll.min(),
}).applymap(lambda x: f"{x:.2%}")

## Checkpoint

**Why would you use log returns instead of simple returns?**

Log returns are *time-additive*: the log return over a span equals the sum of
the daily log returns, whereas simple returns must be compounded
multiplicatively. That makes aggregation, annualizing, and many statistical
operations (which assume additivity) cleaner, and it keeps very small or very
large moves numerically well-behaved. For small daily moves the two are nearly
equal, so the choice mostly matters for math convenience rather than the
headline numbers.